### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [10]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re part of the Psittaciformes order, right? And many species are known for their ability to mimic human speech. But why do they do that? Is it just imitation, or is there a deeper reason?\n\nFirst, maybe I should think about the vocal anatomy of parrots. I remember they have a syrinx instead of a larynx, which is the vocal organ in mammals. The syrinx is located at the base of the trachea where it branches into the bronchi. This structure might give them the ability to produce a wide range of sounds, which could be important for mimicry.\n\nBut why would they evolve that ability? In the wild, parrots might use their vocalizations for communication within their flocks. Maybe mimicking sounds helps them integrate into the group or recognize other members. For example, some parrot species have regional dialects, which suggests that vocal learning is import

In [11]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    # In a real implementation, this would call a weather API.
    return f"Its sunny in {location} ."


model_with_tools = model.bind_tools([get_weather])

In [12]:
response = model_with_tools.invoke("What's the weather like in New York?")
print(response)

for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in New York. I need to use the get_weather function for that. Let me check the function parameters. It requires a location, which is provided as New York. So I\'ll call get_weather with location set to "New York". Make sure the JSON is correctly formatted with the name and arguments. No other parameters are needed. Just the location. Alright, that should do it.\n', 'tool_calls': [{'id': 'w5ncdvzn0', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 156, 'total_tokens': 268, 'completion_time': 0.179109923, 'completion_tokens_details': {'reasoning_tokens': 87}, 'prompt_time': 0.006705502, 'prompt_tokens_details': None, 'queue_time': 0.186052977, 'total_time': 0.185815425}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand'

## Tools execution loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#print(messages)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
# "The current weather in Boston is 72°F and sunny." ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny right now! If you're there, it's a great day to enjoy outdoor activities.


In [14]:
messages 

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'rh5mrx7ej', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.154311124, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006536684, 'prompt_tokens_details': None, 'queue_time': 0.046592287, 'total_time': 0.160847808}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'l